<a href="https://colab.research.google.com/github/nunez1405/Data-science-projects/blob/main/productivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install --upgrade kagglehub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amar5693/screen-time-sleep-and-stress-analysis-dataset")

print("Path to dataset files:", path)

KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.

### Step 1: Generate Kaggle API Token

1.  Go to your Kaggle account page: [https://www.kaggle.com/me/account](https://www.kaggle.com/me/account)
2.  Scroll down to the 'API' section.
3.  Click on 'Create New API Token'. This will download a `kaggle.json` file to your computer. This file contains your username and API key.

### Step 2: Upload `kaggle.json` to Colab

Run the following code cell. It will prompt you to upload the `kaggle.json` file you just downloaded.

In [ ]:
from google.colab import files

files.upload()

### Step 3: Configure Kaggle Environment

Now, move the uploaded `kaggle.json` file to the correct directory and set the appropriate permissions. This allows the Kaggle API to find and use your credentials.

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

### Step 4: Verify Kaggle API Access

You can now try to list Kaggle datasets or download the dataset again to verify that your authentication is working.

In [ ]:
import kaggle

# This should now work without an authentication error
!kaggle datasets list -s "screen time sleep and stress analysis dataset"

# Or re-run your original download command
import kagglehub

path = kagglehub.dataset_download("amar5693/screen-time-sleep-and-stress-analysis-dataset")
print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import matplotlib.ticker  as mticker

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
df = pd.read_csv(os.path.join(path, filename))

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
for column in df.columns:
    print(f"{column}: {df[column].nunique()}")

In [ ]:
df.duplicated().any()


In [ ]:
df.describe()

In [ ]:
anomaly_count = (df['Social_Media_Hours'] > df['Daily_Phone_Hours']).sum()
percentage_anomaly = (anomaly_count/len(df))*100
print(f"Number of anomalies: {anomaly_count}")
print(f"Percentage of anomalies: {percentage_anomaly:.2f}%")

FEATURE ENGINEERING

In [ ]:
df.drop(columns=['User_ID'], inplace=True)


In [ ]:
df['Phone_Addiction_Ratio'] = (df['Daily_Phone_Hours'] / 24).round(4)
df['Weekend_vs_Weekday_Ratio'] = (df['Weekend_Screen_Time_Hours'] / df['Daily_Phone_Hours']).round(4)
df['Social_Media_Pct'] = (df['Social_Media_Hours'] / df['Daily_Phone_Hours']).clip(upper=1.0).round(4)

In [ ]:
def bin_score(score):
    if score <= 3:
        return 'Low'
    elif score <= 7:
        return 'Medium'
    else:
        return 'High'


In [ ]:
df['Productivity_Class'] = df['Work_Productivity_Score'].apply(bin_score)
df['Stress_Class'] = df['Stress_Level'].apply(bin_score)

In [ ]:
class_order = ['Low', 'Medium', 'High']
df['Productivity_Class'] = pd.Categorical(df['Productivity_Class'], categories=class_order, ordered=True)
df['Stress_Class'] = pd.Categorical(df['Stress_Class'], categories=class_order, ordered=True)

print(f'✅ Feature engineering complete. New shape: {df.shape}')
print(f'\nNew columns: Phone_Addiction_Ratio, Weekend_vs_Weekday_Ratio, Social_Media_Pct,')
print(f'             Productivity_Class, Stress_Class, Data_Anomaly')
print(f'\nProductivity Class Distribution:')
print(df['Productivity_Class'].value_counts().sort_index())
print(f'\nStress Class Distribution:')
print(df['Stress_Class'].value_counts().sort_index())
display(df.head())

Visualizacion


In [ ]:
import seaborn.objects as so
df

In [ ]:
(
    so.Plot(df, x='Productivity_Class', color='Productivity_Class') # Add color mapping
    .add(so.Bar(), so.Count())
    .label(
        title='Distribution of Productivity Class',
        color='Productivity Class' # Label for the color legend
    )
    .theme({"axes.grid": True})
)

In [ ]:
(
    so.Plot(df, x='Stress_Class', color='Stress_Class') # Add color mapping
    .add(so.Bar(), so.Count())
    .label(
        title='Distribution of Stress Class',
        color='Stress Class' # Label for the color legend
    )
    .theme({"axes.grid": True})
)

In [ ]:
PALETTE = sns.color_palette('Set2')
CLASS_PAL = {'Low': '#e74c3c', 'Medium': '#f39c12', 'High': '#27ae60'}
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, title in zip(axes, ['Productivity_Class', 'Stress_Class'],
                           ['Work Productivity Class', 'Stress Level Class']):
    order = ['Low', 'Medium', 'High']
    counts = df[col].value_counts().reindex(order)
    total = counts.sum()

    bars = ax.bar(order, counts, color=[CLASS_PAL[c] for c in order], edgecolor='white', linewidth=1.5)

    for bar, count in zip(bars, counts):
        pct = count / total * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.01,
                f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold', fontsize=11)

    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.set_ylim(0, counts.max() * 1.18)

plt.suptitle('📊 Target Variable Distributions', fontsize=17, fontweight='bold', y=1.05) # Adjusted y to 1.05
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Use sns.catplot for a more robust faceted categorical plot
g = sns.catplot(
    data=df,
    x="Productivity_Class",
    col="Gender",
    kind="count", # Specify 'count' for count plot
    hue="Productivity_Class", # Color bars by productivity class within each facet
    order=class_order,
    palette=CLASS_PAL, # Use the custom palette defined earlier
    height=4,
    aspect=1.2,
    sharey=True # Share the y-axis across facets for better comparison
)

# Set labels and titles
g.set_axis_labels("Productivity Class", "Count")
g.set_titles("{col_name}") # Set titles for each facet (e.g., 'Male', 'Female', 'Other')
g.fig.suptitle("Distribution of Productivity Class by Gender", y=1.02) # Set overall figure title

# Adjust subplot parameters for a tighter layout and to prevent title overlap
plt.subplots_adjust(top=0.85, wspace=0.3)
plt.show()

In [ ]:
num_cols = ['Age', 'Daily_Phone_Hours', 'Social_Media_Hours', 'Sleep_Hours',
            'App_Usage_Count', 'Caffeine_Intake_Cups', 'Weekend_Screen_Time_Hours',
            'Phone_Addiction_Ratio', 'Weekend_vs_Weekday_Ratio']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.ravel()

for i, col in enumerate(num_cols):
    ax = axes[i]
    for cls in ['Low', 'Medium', 'High']:
        subset = df[df['Productivity_Class'] == cls][col]
        ax.hist(subset, bins=30, alpha=0.5, label=cls, color=CLASS_PAL[cls], edgecolor='white')
    ax.set_title(col.replace('_', ' '), fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=8, title='Productivity', title_fontsize=8)

plt.suptitle('📈 Numeric Feature Distributions by Productivity Class', fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
import math
ncols = 3
nrows = math.ceil(len(num_cols) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
fig.suptitle("Distribución de variables numéricas", fontsize=16, fontweight="bold")
axes = axes.flatten()

for i, col in enumerate(num_cols):
    (
        so.Plot(df, x=col, color="Productivity_Class")
        .add(so.Bar(), so.Hist(bins=20, stat="density"), so.Stack())
        .label(title=col)
        .theme({"axes.grid": True})
        .on(axes[i])
        .plot()
    )

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()